# cuMotion 튜토리얼 — 평소 실행
기존 EC2 환경을 다시 사용하는 노트북입니다. **전체 실행 대신 순서대로 실행하세요.** 설치·복구 셀은 아래 보관함에 있습니다. 접속 예시의 주소와 키 파일 경로를 본인 환경에 맞게 바꾸세요.

현재 확인된 상태: Isaac Sim 화면과 관절 수신 성공, cuMotion 초기화 성공, 제어 플러그인 소스 빌드 성공. **새 플러그인 적용 후 로봇 계획·실행 성공은 아직 미확인입니다.** 이 노트북을 만들면서 EC2 명령을 실행하지 않았습니다.

1. AWS 콘솔 → EC2 → 인스턴스 → 해당 인스턴스 시작 → 현재 퍼블릭 IPv4 확인.
2. Windows PowerShell에서 기존 SSH 터널을 열고 Jupyter를 시작합니다. 아래 IP는 매번 확인하세요.
3. Colab 오른쪽 위 연결 메뉴 → 로컬 런타임에 연결 → Jupyter 출력 URL의 포트 8888을 9000으로 바꿔 입력합니다. 토큰은 현재 출력값을 사용합니다.

PowerShell 예시:
```powershell
ssh -i "$env:USERPROFILE\.ssh\example-ec2-key.pem" -L 127.0.0.1:9000:127.0.0.1:8888 -L 127.0.0.1:6080:127.0.0.1:6080 -o ExitOnForwardFailure=yes ubuntu@203.0.113.10
```
SSH 연결 후 EC2 터미널에서, Jupyter가 아직 실행되지 않았을 때:
```bash
/home/ubuntu/colab-venv/bin/jupyter notebook --ip=127.0.0.1 --port=8888 --ServerApp.port_retries=0 --ServerApp.allow_origin=https://colab.research.google.com --ServerApp.allow_credentials=True --ServerApp.root_dir=/home/ubuntu/isaac-notebooks
```
이후 코드 셀은 **EC2에 연결된 Colab에서만** 실행합니다.

## 1. 실행 서버 확인 — 저장하세요

In [ ]:
from pathlib import Path
import socket, subprocess
if not Path('/home/ubuntu/workspaces/isaac_ros-dev').is_dir():
    raise RuntimeError('EC2 로컬 런타임에 연결했는지 확인하세요.')
print('서버:', socket.gethostname())
subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv'], check=True)
subprocess.run(['systemctl', 'list-timers', '--all', 'isaac-auto-stop.timer', '--no-pager'], check=True)

## 2. 화면 컨테이너 시작 — 저장하세요
아래 public_ip를 현재 AWS 퍼블릭 IPv4로 수정하세요. IP가 바뀌면 기존 스트리밍 컨테이너를 보관하고 새로 만듭니다. 기존 컨테이너가 있는 현재 환경용 코드입니다.

In [ ]:
import ipaddress
import json
import subprocess
from datetime import datetime

def docker(*args):
    r = subprocess.run(
        ["sudo", "-n", "docker", *args],
        capture_output=True, text=True
    )
    if r.returncode:
        raise RuntimeError(r.stderr or r.stdout)
    return r.stdout.strip()

public_ip = "203.0.113.10"  # 현재 AWS 퍼블릭 IPv4로 수정
ipaddress.IPv4Address(public_ip)

# 지난번 검증한 GPU 라이브러리 목록 갱신
subprocess.run(
    ["sudo", "-n", "systemctl", "restart", "nvidia-cdi-refresh.service"],
    check=True
)

# RViz 원격 화면 시작
docker("start", "cumotion-desktop")

name = "isaac-stream"
old = json.loads(docker("inspect", name))[0]
config = old["Config"]
prefix = "--/exts/omni.kit.livestream.app/primaryStream/publicIp="

saved_args = config.get("Cmd") or []
new_args = [arg for arg in saved_args if not arg.startswith(prefix)]
new_args.append(prefix + public_ip)

if new_args == saved_args:
    docker("start", name)
else:
    # 기존 컨테이너 보관, 캐시와 설정을 재사용
    docker("stop", "--timeout", "30", name)
    backup = name + "-old-ip-" + datetime.now().strftime("%Y%m%d-%H%M%S")
    docker("rename", name, backup)

    cmd = [
        "run", "-d", "--name", name,
        "--runtime=nvidia", "--network=host",
    ]
    if config.get("User"):
        cmd += ["--user", config["User"]]
    if config.get("WorkingDir"):
        cmd += ["--workdir", config["WorkingDir"]]

    for env in config.get("Env", []):
        cmd += ["-e", env]

    for mount in old["Mounts"]:
        if mount["Type"] == "bind":
            mode = "rw" if mount["RW"] else "ro"
            cmd += ["-v", f'{mount["Source"]}:{mount["Destination"]}:{mode}']

    entrypoint = config.get("Entrypoint") or []
    if entrypoint:
        cmd += ["--entrypoint", entrypoint[0]]

    cmd += [old["Image"]] + entrypoint[1:] + new_args
    docker(*cmd)

print("=== 실행 상태 ===")
print(docker("ps", "--format", "{{.Names}} | {{.Status}}"))

print("\n=== 자동 중지까지 남은 시간 ===")
r = subprocess.run(
    ["systemctl", "list-timers", "--all",
     "isaac-auto-stop.timer", "--no-pager"],
    capture_output=True, text=True
)
print(r.stdout + r.stderr)

## 3. 로봇 장면 준비
Windows의 Isaac Sim WebRTC Streaming Client에서 현재 EC2 IP로 연결합니다. Isaac Sim 내부 메뉴 → Window → Examples → Robotics Examples → ROS2 → MoveIt → Franka MoveIt → Load Sample Scene. 이미 장면이 열려 있으면 다시 불러오지 말고 재생 상태를 확인하세요.

RViz는 Chrome 주소창에 `http://127.0.0.1:6080/vnc.html`을 입력해 봅니다. 두 화면은 역할이 다릅니다: Isaac Sim은 시뮬레이션, RViz는 경로 계획 조작입니다.

## 4. 관절 수신 확인 — 필요할 때만
종료 코드 0과 panda 관절 목록이 나와야 합니다. 124이면 시간 초과입니다. 장면 재생 및 UDP 설정을 확인하고 다음 단계로 넘어가지 마세요. 이 코드출력을 복사하고 삭제하셔도 됩니다. 반복 점검용으로 보관해도 됩니다.

In [ ]:
import subprocess
script = r'''
source /opt/ros/jazzy/setup.bash
export RMW_IMPLEMENTATION=rmw_fastrtps_cpp
export FASTRTPS_DEFAULT_PROFILES_FILE=/workspaces/isaac_ros-dev/dds-udp.xml
export FASTDDS_DEFAULT_PROFILES_FILE="$FASTRTPS_DEFAULT_PROFILES_FILE"
timeout 20 ros2 topic echo /isaac_joint_states \
  sensor_msgs/msg/JointState --once \
  --qos-reliability best_effort --qos-durability volatile
'''

r = subprocess.run(
    ["sudo", "-n", "docker", "exec",
     "cumotion-desktop", "bash", "-lc", script],
    capture_output=True, text=True, timeout=35
)
print(r.stdout + r.stderr)
print("종료 코드:", r.returncode)

## 5. 수정 플러그인으로 튜토리얼 시작 — 저장하세요
cumotion-desktop을 재시작하므로 기존 RViz·ROS 작업이 종료됩니다. Isaac Sim은 유지됩니다. UDP 파일, Franka 수정 파일, control-overlay 빌드가 있어야 합니다. **실행 요청 성공은 튜토리얼 성공을 뜻하지 않습니다. 다음 셀로 확인하세요.**

In [ ]:
import subprocess
from pathlib import Path
from datetime import datetime

log_name = "franka-overlay-" + datetime.now().strftime("%Y%m%d-%H%M%S") + ".log"
latest_franka_log = (
    Path.home() / "workspaces/isaac_ros-dev/logs" / log_name
)

subprocess.run([
    'sudo', '-n', 'docker', 'exec', 'cumotion-desktop', 'bash', '-lc',
    'set -e; test -f /workspaces/isaac_ros-dev/dds-udp.xml; '
    'test -f /workspaces/isaac_ros-dev/franka-fix/franka.fixed.launch.py; '
    'test -f /workspaces/isaac_ros-dev/control-overlay/install/setup.bash; '
    'mkdir -p /workspaces/isaac_ros-dev/logs'
], check=True)

subprocess.run(
    ["sudo", "-n", "docker", "restart", "--timeout", "30",
     "cumotion-desktop"],
    check=True
)

script = r'''
set -e
source /opt/ros/jazzy/setup.bash
source /workspaces/isaac_ros-dev/control-overlay/install/setup.bash
ros2 pkg prefix topic_based_ros2_control

export DISPLAY=:99
export LIBGL_ALWAYS_SOFTWARE=1
export RMW_IMPLEMENTATION=rmw_fastrtps_cpp
export FASTRTPS_DEFAULT_PROFILES_FILE=/workspaces/isaac_ros-dev/dds-udp.xml
export FASTDDS_DEFAULT_PROFILES_FILE="$FASTRTPS_DEFAULT_PROFILES_FILE"

# 화면 준비 확인
for i in $(seq 1 30); do
    if xdpyinfo -display :99 >/dev/null 2>&1; then
        break
    fi
    sleep 1
done
xdpyinfo -display :99 >/dev/null

# 처음 열린 빈 RViz 정리
pkill -x rviz2 || true

exec ros2 launch \
    /workspaces/isaac_ros-dev/franka-fix/franka.fixed.launch.py
'''

r = subprocess.run(
    [
        "sudo", "-n", "docker", "exec", "-d",
        "cumotion-desktop", "bash", "-lc",
        f"exec > /workspaces/isaac_ros-dev/logs/{log_name} 2>&1\n" + script
    ],
    capture_output=True, text=True
)
print(r.stdout + r.stderr)
print("실행 요청 종료 코드:", r.returncode)
print("로그:", latest_franka_log)

## 6. 최신 실행 로그 확인 — 필요할 때만
새 플러그인 경로가 control-overlay 아래인지, 제어기 활성화가 되는지 확인합니다. 오류·프로세스 종료가 있으면 실행 성공으로 판단하지 마세요. 이 코드출력을 복사하고 삭제하셔도 됩니다.

In [ ]:
from pathlib import Path
folder = Path.home() / 'workspaces/isaac_ros-dev/logs'
logs = sorted(folder.glob('franka-overlay-*.log'), key=lambda p: p.stat().st_mtime)
if not logs:
    raise RuntimeError('실행 로그가 없습니다. 이전 실행 셀 결과를 확인하세요.')
log = logs[-1]
print('로그:', log)
lines = log.read_text(errors='replace').splitlines()
keys = ('error', 'fatal', 'segmentation', 'died', 'activated', 'initialized', 'success', 'control-overlay')
print('=== 주요 기록 ===')
print('\n'.join(line for line in lines if any(k in line.lower() for k in keys))[-18000:])
print('\n=== 마지막 기록 ===')
print('\n'.join(lines[-45:]))

## 선택: 자동 중지 시간을 지금부터 2시간 연장
시간이 부족할 때만 실행합니다. 서버 사용 비용이 계속 발생합니다. 평소 실행 셀과 별개입니다. 저장해 두세요.

In [ ]:
import subprocess

script = r'''
set -e
sudo -n mkdir -p /run/systemd/system/isaac-auto-stop.timer.d
printf '%s\n' '[Timer]' 'OnBootSec=' 'OnActiveSec=2h' |
  sudo -n tee /run/systemd/system/isaac-auto-stop.timer.d/extend.conf >/dev/null
sudo -n systemctl daemon-reload
sudo -n systemctl restart isaac-auto-stop.timer
systemctl list-timers --all isaac-auto-stop.timer --no-pager
'''

r = subprocess.run(
    ["bash", "-lc", script],
    capture_output=True, text=True
)
print(r.stdout + r.stderr)
print("종료 코드:", r.returncode)

## 종료
Colab 메뉴 → 파일 → 저장. AWS 콘솔 → EC2 → 인스턴스 → 해당 인스턴스 선택 → 인스턴스 상태 → 중지. 중지됨을 확인한 뒤 뷰어·PowerShell을 닫습니다. 종료(삭제)를 선택하지 마세요. EBS 저장 비용은 중지 후에도 남습니다.

---
# 아래는 설치·복구 전용입니다
평소에는 위쪽 실행 부분만 사용하세요. 아래 각 코드 셀은 실행_허용이 False이면 작업 없이 중단됩니다. 필요한 복구 셀만 True로 바꾸고 실행한 뒤 다시 False로 돌려놓으세요. 전체 실행하지 마세요.

In [ ]:
import subprocess

script = r'''
set -e

echo "=== 기존 화면 서버 로그 ==="
tail -n 25 /tmp/vnc.log /tmp/xvfb.log 2>/dev/null || true

echo "=== 가상 화면 확인 ==="
if ! xdpyinfo -display :99 >/dev/null 2>&1; then
    echo "가상 화면 :99에 연결할 수 없습니다. 위 로그를 보내주세요."
    exit 1
fi

echo "=== VNC 화면 서버 복구 ==="
python3 - <<'PY'
import socket
import subprocess
import time
from datetime import datetime

def available():
    try:
        with socket.create_connection(("127.0.0.1", 5900), timeout=1):
            return True
    except OSError:
        return False

if available():
    print("5900 포트가 이미 응답합니다.")
else:
    path = "/tmp/vnc-recovery-" + datetime.now().strftime("%Y%m%d-%H%M%S") + ".log"
    with open(path, "w") as log:
        process = subprocess.Popen(
            ["x11vnc", "-display", ":99", "-localhost",
             "-rfbport", "5900", "-nopw", "-forever", "-shared"],
            stdin=subprocess.DEVNULL,
            stdout=log, stderr=subprocess.STDOUT,
            start_new_session=True
        )

    for _ in range(15):
        if available() or process.poll() is not None:
            break
        time.sleep(1)

    if available():
        print("VNC 화면 서버 복구 완료")
    else:
        print(open(path).read())
        raise RuntimeError("VNC 시작 실패. 위 로그를 보내주세요.")
PY
'''

r = subprocess.run(
    ["sudo", "-n", "docker", "exec", "cumotion-desktop",
     "bash", "-lc", script],
    capture_output=True, text=True, timeout=35
)
print(r.stdout + r.stderr)
print("종료 코드:", r.returncode)

In [ ]:
import subprocess

patch = r'''
from pathlib import Path
from datetime import datetime
import shutil

path = Path("/workspaces/isaac_ros-dev/start-desktop.sh")
text = path.read_text()
marker = "# DISPLAY_99_STARTUP_FIX"

if marker in text:
    print("화면 시작 복구가 이미 적용되어 있습니다.")
else:
    anchor = "Xvfb :99 "
    assert text.count(anchor) == 1, "예상과 다른 시작 스크립트입니다."

    preparation = r"""
# DISPLAY_99_STARTUP_FIX
# 실제 화면 서버가 없을 때만 이전 실행의 잠금 파일을 정리합니다.
if xdpyinfo -display :99 >/dev/null 2>&1; then
    echo "이미 가상 화면이 실행 중입니다. 중복 실행을 중단합니다."
    exit 1
fi
if pgrep -x Xvfb >/dev/null; then
    echo "Xvfb 프로세스가 남아 있습니다. 잠금 파일을 변경하지 않습니다."
    exit 1
fi
rm -f -- /tmp/.X99-lock /tmp/.X11-unix/X99
"""

    text = text.replace(anchor, preparation + "\n" + anchor, 1)

    old_wait = "sleep 2"
    assert text.count(old_wait) == 1, "예상과 다른 화면 대기 코드입니다."
    new_wait = r"""
# 소켓 파일 존재가 아닌 실제 화면 응답을 확인합니다.
for i in $(seq 1 30); do
    if xdpyinfo -display :99 >/dev/null 2>&1; then
        break
    fi
    sleep 1
done
if ! xdpyinfo -display :99 >/dev/null 2>&1; then
    cat /tmp/xvfb.log
    exit 1
fi
"""
    text = text.replace(old_wait, new_wait, 1)

    backup = path.with_name(
        "start-desktop.before-display-fix-"
        + datetime.now().strftime("%Y%m%d-%H%M%S") + ".sh"
    )
    shutil.copy2(path, backup)
    path.write_text(text)
    print("백업:", backup)
    print("화면 시작 스크립트 수정 완료")
'''

r = subprocess.run(
    ["sudo", "-n", "docker", "exec", "-i", "-u", "root",
     "cumotion-desktop", "python3", "-"],
    input=patch, capture_output=True, text=True
)
print(r.stdout + r.stderr)
print("종료 코드:", r.returncode)

# 설치·복구 보관함
**매번 실행하지 않습니다. 전체 실행 금지.** 기존 EC2 환경에서 필요한 항목만 실행하세요. 원본의 설치 절차를 보관한 것으로 새 서버 전체 자동 구축을 보장하지 않습니다. 컨테이너 내부 설치는 컨테이너를 삭제·재생성하면 사라질 수 있습니다. 작업 폴더의 수정 파일과 overlay는 호스트에 남습니다. 패키지 저장소는 고정 스냅샷이 아니므로 미래 빌드는 달라질 수 있습니다.

In [ ]:
import subprocess

script = r'''
set -e
source /opt/ros/jazzy/setup.bash
source /workspaces/isaac_ros-dev/control-overlay/install/setup.bash

python3 - <<'PY'
from pathlib import Path
import subprocess
import shutil
import xml.etree.ElementTree as ET
import yaml

root = Path("/workspaces/isaac_ros-dev")
commit = "0f8cc787bdf6c383aba22158c43fdbece51f3a61"
repo = root / "sources/cumotion-v4.6-0"
repo.parent.mkdir(parents=True, exist_ok=True)

def run(*args):
    subprocess.run(args, check=True)

if not repo.exists():
    run("git", "clone", "--depth", "1", "--branch", "v4.6-0",
        "https://github.com/NVIDIA-ISAAC-ROS/isaac_ros_cumotion.git",
        str(repo))

actual = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
).strip()
assert actual == commit, "예상한 릴리스 커밋과 다릅니다. 중단합니다."

source = repo / "isaac_ros_cumotion_moveit"
version = ET.parse(source / "package.xml").getroot().findtext("version")
assert version == "4.6.0", f"예상과 다른 패키지 버전: {version}"

xrdf = yaml.safe_load(
    (root / "franka-fix/franka.fixed.xrdf").read_text()
)
names = xrdf["cspace"]["joint_names"]
assert names == [f"panda_joint{i}" for i in range(1, 8)], \
    "예상과 다른 팔 관절 구성입니다."

dest = root / "control-overlay/src/isaac_ros_cumotion_moveit"
if not dest.exists():
    shutil.copytree(source, dest)

path = dest / "src/cumotion_move_group_client.cpp"
original = (source / "src/cumotion_move_group_client.cpp").read_text()
current = path.read_text()
marker = "// PANDA_ARM_REQUEST_FIX_V1"

if marker in current:
    print("같은 수정이 이미 적용되어 있습니다.")
else:
    assert current == original, "기존 사용자 수정이 있습니다. 덮어쓰지 않습니다."
    anchor = "  planning_request_ = req;"
    assert original.count(anchor) == 1, \
        "요청 전달 코드가 예상과 다릅니다. 수정하지 않습니다."

    patch = r"""
  // PANDA_ARM_REQUEST_FIX_V1
  // 팔 계획 요청만 XRDF 순서의 7개 관절로 정리합니다.
  if (planning_request_.group_name == "panda_arm") {
    const std::vector<std::string> arm_names = {
      "panda_joint1", "panda_joint2", "panda_joint3",
      "panda_joint4", "panda_joint5", "panda_joint6", "panda_joint7"
    };

    const auto input = planning_request_.start_state.joint_state;
    if (input.name.size() != input.position.size()) {
      throw std::runtime_error("시작 관절 이름과 위치 개수가 다릅니다.");
    }

    auto filtered = input;
    filtered.name.clear();
    filtered.position.clear();
    filtered.velocity.clear();
    filtered.effort.clear();

    for (const auto & name : arm_names) {
      const auto it = std::find(input.name.begin(), input.name.end(), name);
      if (it == input.name.end()) {
        throw std::runtime_error("시작 상태에 필요한 팔 관절이 없습니다.");
      }
      const auto i = static_cast<size_t>(std::distance(input.name.begin(), it));
      filtered.name.push_back(name);
      filtered.position.push_back(input.position.at(i));
      if (input.velocity.size() == input.name.size()) {
        filtered.velocity.push_back(input.velocity.at(i));
      }
      if (input.effort.size() == input.name.size()) {
        filtered.effort.push_back(input.effort.at(i));
      }
    }
    planning_request_.start_state.joint_state = filtered;
    // 전달되는 값은 현재 관절의 절대 위치입니다.
    planning_request_.start_state.is_diff = false;

    for (auto & goal : planning_request_.goal_constraints) {
      if (goal.joint_constraints.empty()) {
        continue;
      }
      const auto constraints = goal.joint_constraints;
      goal.joint_constraints.clear();
      for (const auto & name : arm_names) {
        const auto it = std::find_if(
          constraints.begin(), constraints.end(),
          [&name](const auto & item) { return item.joint_name == name; });
        if (it == constraints.end()) {
          throw std::runtime_error("목표 상태에 필요한 팔 관절이 없습니다.");
        }
        goal.joint_constraints.push_back(*it);
      }
    }

    RCLCPP_INFO(
      node_->get_logger(), "팔 계획 요청 정리 완료: %zu -> %zu",
      input.position.size(), filtered.position.size());
  }
"""
    text = original.replace(anchor, anchor + "\n" + patch)
    text = (
        "#include <algorithm>\n"
        "#include <iterator>\n"
        "#include <stdexcept>\n"
        "#include <vector>\n"
        + text
    )
    path.with_suffix(".cpp.before-arm-fix").write_text(original)
    path.write_text(text)
    print("팔 계획 요청 수정 완료:", path)

print("소스 커밋:", actual)
print("패키지 버전:", version)
print("팔 관절 순서:", names)
PY

cd /workspaces/isaac_ros-dev/control-overlay

colcon build \
  --packages-select isaac_ros_cumotion_moveit \
  --allow-overriding isaac_ros_cumotion_moveit \
  --executor sequential \
  --cmake-args -DCMAKE_BUILD_TYPE=Release -DBUILD_TESTING=OFF \
  --event-handlers console_direct+

source install/setup.bash
echo "=== 수정 플러그인 설치 위치 ==="
ros2 pkg prefix isaac_ros_cumotion_moveit
'''

p = subprocess.Popen(
    ["sudo", "-n", "docker", "exec", "-u", "root",
     "cumotion-desktop", "bash", "-lc", script],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
for line in p.stdout:
    print(line, end="", flush=True)

code = p.wait()
print("\n종료 코드:", code)

## Isaac ROS CLI 설치: 도구가 없을 때만
복구 절차로 저장하세요. 평소에는 실행하지 않습니다. 원본 코드 셀 번호: 16

In [ ]:
실행_허용 = False  # 이 복구가 필요한 경우에만 True로 변경
if not 실행_허용:
    raise RuntimeError('복구 작업을 실행하지 않았습니다. 필요한 셀만 실행_허용을 True로 바꾸세요.')

import subprocess

script = r'''
set -euo pipefail

sudo -n apt-get update -qq
sudo -n apt-get install -y --no-install-recommends \
  ca-certificates curl gnupg

# NVIDIA 공식 release-4.6 저장소 등록
temp_key=$(mktemp)
curl -fsSL https://isaac.download.nvidia.com/isaac-ros/repos.key \
  | gpg --batch --dearmor > "$temp_key"
sudo -n install -m 644 "$temp_key" /usr/share/keyrings/nvidia-isaac-ros.gpg
rm -f "$temp_key"

printf '%s\n' \
  'deb [signed-by=/usr/share/keyrings/nvidia-isaac-ros.gpg] https://isaac.download.nvidia.com/isaac-ros/release-4.6 noble main' \
  | sudo -n tee /etc/apt/sources.list.d/nvidia-isaac-ros.list >/dev/null

sudo -n apt-get update -qq
sudo -n env DEBIAN_FRONTEND=noninteractive \
  apt-get install -y --no-remove isaac-ros-cli

echo "=== 설치 도구 버전 ==="
dpkg-query -W isaac-ros-cli

echo "=== Docker 초기화 옵션 ==="
isaac-ros init --help
'''

process = subprocess.Popen(
    ["bash", "-lc", script],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
for line in process.stdout:
    print(line, end="", flush=True)
print("\n종료 코드:", process.wait())

## CLI Docker 모드 설정: 초기 환경 구성 때만
복구 절차로 저장하세요. 평소에는 실행하지 않습니다. 원본 코드 셀 번호: 17

In [ ]:
실행_허용 = False  # 이 복구가 필요한 경우에만 True로 변경
if not 실행_허용:
    raise RuntimeError('복구 작업을 실행하지 않았습니다. 필요한 셀만 실행_허용을 True로 바꾸세요.')

import subprocess

script = r'''
set -e
sudo -n isaac-ros init docker --yes

echo "=== 환경 실행 옵션 ==="
isaac-ros activate --help

echo "=== 자동 중지까지 남은 시간 ==="
systemctl list-timers --all isaac-auto-stop.timer --no-pager
'''

process = subprocess.Popen(
    ["bash", "-lc", script],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
for line in process.stdout:
    print(line, end="", flush=True)
print("\n종료 코드:", process.wait())

## ROS 기본 이미지 준비: 이미지가 없을 때만
복구 절차로 저장하세요. 평소에는 실행하지 않습니다. 원본 코드 셀 번호: 18

In [ ]:
실행_허용 = False  # 이 복구가 필요한 경우에만 True로 변경
if not 실행_허용:
    raise RuntimeError('복구 작업을 실행하지 않았습니다. 필요한 셀만 실행_허용을 True로 바꾸세요.')

import subprocess

script = r'''
set -e

# 작업 폴더 준비
export ISAAC_ROS_WS="$HOME/workspaces/isaac_ros-dev"
mkdir -p "$ISAAC_ROS_WS/src"

# Docker 권한을 현재 실행에도 반영
sudo -n usermod -aG docker "$(id -un)"

# 컨테이너에 들어가지 않고 이미지 준비까지만 수행
cd "$ISAAC_ROS_WS"
sg docker -c 'isaac-ros activate --build-only'

echo "=== 준비된 이미지 ==="
sudo -n docker images --format '{{.Repository}}:{{.Tag}}  {{.Size}}'

echo "=== 남은 디스크 ==="
df -h /
'''

process = subprocess.Popen(
    ["bash", "-lc", script],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
for line in process.stdout:
    print(line, end="", flush=True)
print("\n종료 코드:", process.wait())

## cuMotion 이미지 빌드: 이미지가 없거나 의도적으로 재빌드할 때
복구 절차로 저장하세요. 평소에는 실행하지 않습니다. 원본 코드 셀 번호: 20

In [ ]:
실행_허용 = False  # 이 복구가 필요한 경우에만 True로 변경
if not 실행_허용:
    raise RuntimeError('복구 작업을 실행하지 않았습니다. 필요한 셀만 실행_허용을 True로 바꾸세요.')

from pathlib import Path
import subprocess

folder = Path.home() / "workspaces/isaac_ros-dev/docker/cumotion-tutorial"
folder.mkdir(parents=True, exist_ok=True)

# 방금 다운로드한 이미지의 정확한 버전 사용
base = (
    "nvcr.io/nvidia/isaac/ros@sha256:"
    "d9ed7b0d57a9ed1593d1503c31120227ed4aa513716ebbf935b7f7c6b96032e1"
)

dockerfile = f"""
FROM {base}
USER root
ARG DEBIAN_FRONTEND=noninteractive

RUN apt-get update && \\
    apt-get install -y --no-remove \\
      ros-jazzy-isaac-ros-cumotion-examples && \\
    rm -rf /var/lib/apt/lists/*

RUN /bin/bash -c 'source /opt/ros/jazzy/setup.bash && \\
    ros2 pkg prefix isaac_ros_cumotion && \\
    ros2 pkg prefix isaac_ros_cumotion_moveit && \\
    ros2 pkg prefix isaac_ros_cumotion_examples && \\
    ros2 pkg prefix rviz2'
"""

(folder / "Dockerfile").write_text(dockerfile)

process = subprocess.Popen(
    [
        "sudo", "-n", "docker", "build",
        "--progress=plain",
        "-t", "cumotion-tutorial:jazzy",
        str(folder),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

log_path = folder / "build.log"
with log_path.open("w") as log:
    for line in process.stdout:
        log.write(line)
        print(line, end="", flush=True)

code = process.wait()
print("\n종료 코드:", code)
print("설치 로그:", log_path)
if code == 0:
    print("cuMotion 튜토리얼 이미지 생성 완료")

## RViz 데스크톱 이미지 빌드: 이미지가 없을 때
복구 절차로 저장하세요. 평소에는 실행하지 않습니다. 원본 코드 셀 번호: 22

In [ ]:
실행_허용 = False  # 이 복구가 필요한 경우에만 True로 변경
if not 실행_허용:
    raise RuntimeError('복구 작업을 실행하지 않았습니다. 필요한 셀만 실행_허용을 True로 바꾸세요.')

from pathlib import Path
import subprocess

folder = Path.home() / "workspaces/isaac_ros-dev/docker/cumotion-desktop"
folder.mkdir(parents=True, exist_ok=True)

(folder / "Dockerfile").write_text("""
FROM cumotion-tutorial:jazzy
USER root
ARG DEBIAN_FRONTEND=noninteractive

RUN apt-get update && \\
    apt-get install -y --no-install-recommends \\
      xvfb x11vnc novnc websockify openbox xterm mesa-utils && \\
    rm -rf /var/lib/apt/lists/*
""")

process = subprocess.Popen(
    [
        "sudo", "-n", "docker", "build",
        "--progress=plain",
        "-t", "cumotion-desktop:jazzy",
        str(folder),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

with (folder / "build.log").open("w") as log:
    for line in process.stdout:
        log.write(line)
        print(line, end="", flush=True)

print("\n종료 코드:", process.wait())

r = subprocess.run(
    ["systemctl", "list-timers", "--all",
     "isaac-auto-stop.timer", "--no-pager"],
    capture_output=True, text=True
)
print(r.stdout + r.stderr)


## RViz 데스크톱 실행 파일·컨테이너 복구: 컨테이너가 없을 때
복구 절차로 저장하세요. 평소에는 실행하지 않습니다. 원본 코드 셀 번호: 23

In [ ]:
실행_허용 = False  # 이 복구가 필요한 경우에만 True로 변경
if not 실행_허용:
    raise RuntimeError('복구 작업을 실행하지 않았습니다. 필요한 셀만 실행_허용을 True로 바꾸세요.')

from pathlib import Path
import subprocess

folder = Path.home() / "workspaces/isaac_ros-dev"
startup = folder / "start-desktop.sh"
startup.write_text(r'''#!/bin/bash
set -e
source /opt/ros/jazzy/setup.bash
export DISPLAY=:99
export LIBGL_ALWAYS_SOFTWARE=1

Xvfb :99 -screen 0 1280x800x24 -nolisten tcp > /tmp/xvfb.log 2>&1 &
for i in $(seq 1 30); do
    [ -S /tmp/.X11-unix/X99 ] && break
    sleep 1
done
sleep 2

openbox > /tmp/openbox.log 2>&1 &
x11vnc -display :99 -localhost -rfbport 5900 \
    -nopw -forever -shared > /tmp/vnc.log 2>&1 &
rviz2 > /tmp/rviz.log 2>&1 &

exec websockify --web=/usr/share/novnc \
    127.0.0.1:6080 127.0.0.1:5900
''')

def docker(*args):
    r = subprocess.run(
        ["sudo", "-n", "docker", *args],
        capture_output=True, text=True
    )
    if r.returncode:
        raise RuntimeError(r.stderr or r.stdout)
    return r.stdout.strip()

name = "cumotion-desktop"
existing = docker(
    "ps", "-a", "--filter", f"name=^/{name}$",
    "--format", "{{.Names}}"
)
if existing:
    docker("start", name)
else:
    docker(
        "run", "-d", "--name", name,
        "--network=host",
        "--runtime=nvidia",
        "-e", "NVIDIA_VISIBLE_DEVICES=nvidia.com/gpu=all",
        "-e", "NVIDIA_DRIVER_CAPABILITIES=all",
        "-v", f"{folder}:/workspaces/isaac_ros-dev",
        "--entrypoint", "/bin/bash",
        "cumotion-desktop:jazzy",
        "/workspaces/isaac_ros-dev/start-desktop.sh"
    )

print("RViz 화면 컨테이너")
print(docker("ps", "-a", "--filter", f"name=^/{name}$",
             "--format", "{{.Names}} | {{.Status}}"))

## Franka XRDF mimic 관절 오류 및 제어 플러그인 누락 복구
복구 절차로 저장하세요. 평소에는 실행하지 않습니다. 원본 코드 셀 번호: 29

In [ ]:
실행_허용 = False  # 이 복구가 필요한 경우에만 True로 변경
if not 실행_허용:
    raise RuntimeError('복구 작업을 실행하지 않았습니다. 필요한 셀만 실행_허용을 True로 바꾸세요.')

import subprocess

script = r'''
set -e
export DEBIAN_FRONTEND=noninteractive

apt-get install -y --no-remove ros-jazzy-topic-based-ros2-control

python3 - <<'PY'
from pathlib import Path
import xml.etree.ElementTree as ET
import yaml
import shutil
import ast

root = Path("/workspaces/isaac_ros-dev/franka-fix")
root.mkdir(parents=True, exist_ok=True)

xrdf_src = Path(
    "/opt/ros/jazzy/share/isaac_ros_cumotion_robot_description/xrdf/franka.xrdf"
)
launch_src = Path(
    "/opt/ros/jazzy/share/isaac_ros_cumotion_examples/launch/franka_isaac_sim.launch.py"
)
urdf = Path(
    "/opt/ros/jazzy/share/moveit_resources_panda_description/urdf/panda.urdf"
)

# 설치된 URDF에서도 해당 관절이 mimic인지 확인
joint = ET.parse(urdf).getroot().find("./joint[@name='panda_finger_joint2']")
assert joint is not None and joint.find("mimic") is not None, \
    "예상과 다른 URDF입니다. 수정하지 않습니다."

shutil.copy2(xrdf_src, root / "franka.original.xrdf")
shutil.copy2(launch_src, root / "franka.original.launch.py")

data = yaml.safe_load(xrdf_src.read_text())
positions = data["default_joint_positions"]
assert "panda_finger_joint2" in positions
del positions["panda_finger_joint2"]

xrdf_out = root / "franka.fixed.xrdf"
xrdf_out.write_text(yaml.safe_dump(data, sort_keys=False))

text = launch_src.read_text()
target = "'xrdf_file_path': xrdf_path"
assert text.count(target) == 1, "실행 설정 형식이 예상과 다릅니다."
text = text.replace(target, f"'xrdf_file_path': {str(xrdf_out)!r}")
ast.parse(text)
(root / "franka.fixed.launch.py").write_text(text)

print("Franka 수정본 저장 완료:", root)
print("원본 파일은 그대로 보존했습니다.")
PY

source /opt/ros/jazzy/setup.bash
echo "=== 제어 플러그인 확인 ==="
ros2 pkg prefix topic_based_ros2_control

echo "=== 수정된 실행 설정 검사 ==="
ros2 launch /workspaces/isaac_ros-dev/franka-fix/franka.fixed.launch.py --show-args
'''

p = subprocess.Popen(
    ["sudo", "-n", "docker", "exec", "-u", "root",
     "cumotion-desktop", "bash", "-lc", script],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in p.stdout:
    print(line, end="", flush=True)
print("\n종료 코드:", p.wait())

## 제어 플러그인 ABI 충돌 복구: ros2_control 충돌 또는 환경 변경 후 재빌드
복구 절차로 저장하세요. 평소에는 실행하지 않습니다. 원본 코드 셀 번호: 32

In [ ]:
실행_허용 = False  # 이 복구가 필요한 경우에만 True로 변경
if not 실행_허용:
    raise RuntimeError('복구 작업을 실행하지 않았습니다. 필요한 셀만 실행_허용을 True로 바꾸세요.')

import subprocess

script = r'''
set -e
export DEBIAN_FRONTEND=noninteractive

apt-get update -qq
apt-get install -y --no-remove --no-install-recommends \
  git build-essential cmake python3-colcon-common-extensions \
  ros-jazzy-ament-cmake ros-jazzy-angles

source /opt/ros/jazzy/setup.bash
ws=/workspaces/isaac_ros-dev/control-overlay
mkdir -p "$ws/src"
cd "$ws"

if [ ! -d src/topic_based_ros2_control ]; then
    git clone \
      https://github.com/PickNikRobotics/topic_based_ros2_control.git \
      src/topic_based_ros2_control
fi

expected=aa9f9ef9525a60f8becea322537dd1d6d1bf9123
actual=$(git -C src/topic_based_ros2_control rev-parse HEAD)
if [ "$actual" != "$expected" ]; then
    echo "소스 버전이 기존 검증 버전과 다릅니다. 자동 변경하지 않고 중단합니다."
    echo "기대 버전: $expected / 현재 버전: $actual"
    exit 1
fi
echo "=== 빌드에 사용할 소스 버전 ==="
git -C src/topic_based_ros2_control rev-parse HEAD \
  | tee source-commit.txt

colcon build \
  --packages-select topic_based_ros2_control \
  --executor sequential \
  --cmake-args -DCMAKE_BUILD_TYPE=Release -DBUILD_TESTING=OFF \
  --event-handlers console_direct+

source install/setup.bash

echo "=== 새 플러그인 경로 ==="
ros2 pkg prefix topic_based_ros2_control
'''

# 안내 문구를 한국어로 표시


p = subprocess.Popen(
    ["sudo", "-n", "docker", "exec", "-u", "root",
     "cumotion-desktop", "bash", "-lc", script],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in p.stdout:
    print(line, end="", flush=True)
print("\n종료 코드:", p.wait())

## UDP 설정 파일 복구
발행자가 보이지만 관절 데이터를 받지 못했던 문제를 해결한 설정입니다. 파일이 없거나 손상된 경우에만 생성합니다. **나중에도 필요한 파일이며 삭제하지 마세요.** 평소 실행 노트북에서 이 파일을 사용합니다.

In [ ]:
실행_허용 = False  # 이 복구가 필요한 경우에만 True로 변경
if not 실행_허용:
    raise RuntimeError('복구 작업을 실행하지 않았습니다. 필요한 셀만 실행_허용을 True로 바꾸세요.')

from pathlib import Path
import subprocess

folder = Path.home() / "workspaces/isaac_ros-dev"
profile = folder / "dds-udp.xml"
profile.write_text("""<?xml version="1.0" encoding="UTF-8"?>
<profiles xmlns="http://www.eprosima.com/XMLSchemas/fastRTPS_Profiles">
  <transport_descriptors>
    <transport_descriptor>
      <transport_id>udp_transport</transport_id>
      <type>UDPv4</type>
    </transport_descriptor>
  </transport_descriptors>
  <participant profile_name="udp_only" is_default_profile="true">
    <rtps>
      <userTransports>
        <transport_id>udp_transport</transport_id>
      </userTransports>
      <useBuiltinTransports>false</useBuiltinTransports>
    </rtps>
  </participant>
</profiles>
""")



## GPU 영상 라이브러리 목록 갱신 및 검사
호스트 영상 라이브러리는 설치되어 있으나 컨테이너에서 못 찾을 때 사용합니다. 드라이버·영상 패키지 설치 후 CDI 목록을 갱신합니다. 기존 컨테이너 재생성이 필요할 수 있습니다. 이 검사만으로 기존 컨테이너가 복구되었다고 판단하지 마세요.

In [ ]:
실행_허용 = False  # 이 복구가 필요한 경우에만 True로 변경
if not 실행_허용:
    raise RuntimeError('복구 작업을 실행하지 않았습니다. 필요한 셀만 실행_허용을 True로 바꾸세요.')

import subprocess

script = r'''
set -e

echo "=== GPU 라이브러리 목록 갱신 ==="
sudo -n systemctl restart nvidia-cdi-refresh.service

echo "=== 갱신된 목록의 영상 라이브러리 ==="
sudo -n grep -E 'libnvidia-encode|libnvcuvid' /var/run/cdi/nvidia.yaml

echo "=== 임시 컨테이너에서 직접 로딩 검사 ==="
sudo -n docker run --rm \
  --runtime=nvidia \
  -e NVIDIA_VISIBLE_DEVICES=nvidia.com/gpu=all \
  -e NVIDIA_DRIVER_CAPABILITIES=all \
  --entrypoint /isaac-sim/python.sh \
  nvcr.io/nvidia/isaac-sim:6.2.0 \
  -c 'import ctypes; ctypes.CDLL("libnvidia-encode.so.1"); ctypes.CDLL("libnvcuvid.so.1"); print("영상 라이브러리 2개 로딩 성공")'
'''

r = subprocess.run(
    ["bash", "-lc", script],
    capture_output=True, text=True
)
print(r.stdout + r.stderr)
print("종료 코드:", r.returncode)

ready ↔ extended를 자동으로 3회 왕복하는 시연 코드

자동화 수정·검증용

정리본으로 REPEAT = 1을 한 번 확인한 뒤, 3으로 바꾸면 3회 왕복합니다

In [ ]:
import subprocess

# 1회로 확인한 뒤 시연할 때 3으로 변경하세요.
REPEAT = 3

robot_script = r'''
import time, copy, fcntl
import xml.etree.ElementTree as ET
import rclpy
from rclpy.action import ActionClient
from rclpy.qos import qos_profile_sensor_data
from rcl_interfaces.srv import GetParameters
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup, ExecuteTrajectory
from moveit_msgs.msg import Constraints, JointConstraint

REPEAT = __REPEAT__
ARM = [f"panda_joint{i}" for i in range(1, 8)]
lock = open("/tmp/cumotion-roundtrip.lock", "w")
fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
rclpy.init()
node = rclpy.create_node("cumotion_roundtrip_demo")
active = None
states = {}

def receive(key, msg):
    states[key] = (time.monotonic(), msg)

subs = [
    node.create_subscription(
        JointState, topic, lambda msg, key=key: receive(key, msg),
        qos_profile_sensor_data)
    for key, topic in [
        ("control", "/joint_states"), ("sim", "/isaac_joint_states")]
]

def wait(future, seconds):
    end = time.monotonic() + seconds
    while not future.done() and time.monotonic() < end:
        rclpy.spin_once(node, timeout_sec=0.1)
    if not future.done():
        raise TimeoutError("응답 시간 초과")
    return future.result()

def fresh(key, seconds=10):
    since = time.monotonic()
    while time.monotonic() - since < seconds:
        rclpy.spin_once(node, timeout_sec=0.1)
        record = states.get(key)
        if record and record[0] >= since:
            msg = record[1]
            if len(msg.name) == len(msg.position) and all(n in msg.name for n in ARM):
                return copy.deepcopy(msg)
    raise RuntimeError(f"{key}: 새 관절 데이터 수신 실패")

def action(client, goal):
    global active
    active = wait(client.send_goal_async(goal), 15)
    if not active.accepted:
        active = None
        raise RuntimeError("요청이 거절됐습니다.")
    result = wait(active.get_result_async(), 90)
    active = None
    if result.status != 4 or result.result.error_code.val != 1:
        raise RuntimeError(
            f"요청 실패: 상태={result.status}, 코드={result.result.error_code.val}")
    return result.result

try:
    service = node.create_client(GetParameters, "/move_group/get_parameters")
    if not service.wait_for_service(timeout_sec=10):
        raise RuntimeError("MoveIt 설정 서버가 없습니다.")
    query = GetParameters.Request()
    query.names = ["robot_description_semantic"]
    response = wait(service.call_async(query), 10)
    root = ET.fromstring(response.values[0].string_value)
    targets = {}
    for state in root.findall("group_state"):
        if state.get("group") == "panda_arm" and state.get("name") in ("ready", "extended"):
            values = {j.get("name"): float(j.get("value")) for j in state.findall("joint")}
            if not all(n in values for n in ARM):
                raise RuntimeError("저장 자세에 팔 관절이 누락됐습니다.")
            targets[state.get("name")] = values
    if set(targets) != {"ready", "extended"}:
        raise RuntimeError("ready 또는 extended 자세가 없습니다.")
    # 검증된 시연용 목표. 원본 저장 자세는 변경하지 않습니다.
    targets["extended"]["panda_joint4"] = -0.20

    planner = ActionClient(node, MoveGroup, "/move_action")
    executor = ActionClient(node, ExecuteTrajectory, "/execute_trajectory")
    if not planner.wait_for_server(timeout_sec=15) or not executor.wait_for_server(timeout_sec=15):
        raise RuntimeError("계획 또는 실행 서버가 준비되지 않았습니다.")
    fresh("sim")
    print(f"cuMotion 자동 왕복 시작: {REPEAT}회 / 속도·가속도 0.10", flush=True)

    for cycle in range(1, REPEAT + 1):
        for name in ("extended", "ready"):
            goal = MoveGroup.Goal()
            req = goal.request
            req.group_name = "panda_arm"
            req.pipeline_id = "isaac_ros_cumotion"
            req.planner_id = "cuMotion"
            req.num_planning_attempts = 1
            req.allowed_planning_time = 10.0
            req.max_velocity_scaling_factor = 0.10
            req.max_acceleration_scaling_factor = 0.10
            req.start_state.joint_state = fresh("control")
            req.start_state.is_diff = False
            fresh("sim")
            constraint = Constraints()
            constraint.name = name
            for joint in ARM:
                item = JointConstraint()
                item.joint_name = joint
                item.position = targets[name][joint]
                item.tolerance_above = item.tolerance_below = 0.001
                item.weight = 1.0
                constraint.joint_constraints.append(item)
            req.goal_constraints = [constraint]
            goal.planning_options.plan_only = True
            goal.planning_options.replan = False
            goal.planning_options.planning_scene_diff.is_diff = True
            goal.planning_options.planning_scene_diff.robot_state.is_diff = True

            print(f"[{cycle}/{REPEAT}] {name}: 계획 중", flush=True)
            planned = action(planner, goal)
            trajectory = planned.planned_trajectory
            jt = trajectory.joint_trajectory
            if not jt.points or len(jt.joint_names) != len(jt.points[-1].positions):
                raise RuntimeError("유효한 경로가 없습니다.")
            endpoint = dict(zip(jt.joint_names, jt.points[-1].positions))
            if not all(n in endpoint for n in ARM):
                raise RuntimeError("경로 관절이 누락됐습니다.")
            if max(abs(endpoint[n] - targets[name][n]) for n in ARM) > 0.02:
                raise RuntimeError("경로 끝점이 요청한 목표와 다릅니다.")
            print(f"  cuMotion 경로 확인: {len(jt.points)}개 경로점 / 실행 중", flush=True)
            execute = ExecuteTrajectory.Goal()
            execute.trajectory = trajectory
            action(executor, execute)

            deadline = time.monotonic() + 20
            while True:
                msg = fresh("sim", 3)
                values = dict(zip(msg.name, msg.position))
                error = max(abs(values[n] - targets[name][n]) for n in ARM)
                if error <= 0.05:
                    break
                if time.monotonic() >= deadline:
                    raise RuntimeError(f"Isaac Sim 목표 미도달: 최대 오차 {error:.4f} rad")
            print(f"  {name} 도착 확인: 최대 오차 {error:.4f} rad", flush=True)
            time.sleep(2)
    print(f"왕복 {REPEAT}회 완료! 마지막 자세: ready", flush=True)
finally:
    if active is not None and active.accepted:
        try:
            wait(active.cancel_goal_async(), 5)
            print("진행 중 요청에 취소를 요청했습니다.", flush=True)
        except Exception as error:
            print("취소 확인 실패:", error, flush=True)
    node.destroy_node()
    rclpy.shutdown()
    lock.close()
'''
assert isinstance(REPEAT, int) and 1 <= REPEAT <= 3, "반복 횟수는 1~3으로 설정하세요."
robot_script = robot_script.replace("__REPEAT__", str(REPEAT))

command = r'''
set -e
source /opt/ros/jazzy/setup.bash
source /workspaces/isaac_ros-dev/control-overlay/install/setup.bash
export RMW_IMPLEMENTATION=rmw_fastrtps_cpp
export FASTRTPS_DEFAULT_PROFILES_FILE=/workspaces/isaac_ros-dev/dds-udp.xml
export FASTDDS_DEFAULT_PROFILES_FILE="$FASTRTPS_DEFAULT_PROFILES_FILE"
exec timeout --signal=INT --kill-after=10s 600s python3 -u -
'''
p = subprocess.Popen(
    ["sudo", "-n", "docker", "exec", "-i", "cumotion-desktop", "bash", "-lc", command],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True)
p.stdin.write(robot_script)
p.stdin.close()
for line in p.stdout:
    print(line, end="", flush=True)
print("\n종료 코드:", p.wait())


## 상하좌우 각 방향으로 3cm씩 자동으로 이동하는 코드

In [ ]:
import subprocess

# 1회로 확인한 뒤 시연할 때 3으로 변경하세요.
REPEAT = 1
DISTANCE_M = 0.03  # 각 방향 이동폭 3cm

robot_script = r'''
import time, copy, fcntl
import math, yaml
from pathlib import Path
from moveit_msgs.srv import GetPositionFK
from moveit_msgs.msg import PositionConstraint, OrientationConstraint
from shape_msgs.msg import SolidPrimitive
import xml.etree.ElementTree as ET
import rclpy
from rclpy.action import ActionClient
from rclpy.qos import qos_profile_sensor_data
from rcl_interfaces.srv import GetParameters
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup, ExecuteTrajectory
from moveit_msgs.msg import Constraints, JointConstraint

REPEAT = __REPEAT__
DISTANCE = __DISTANCE__
ARM = [f"panda_joint{i}" for i in range(1, 8)]
lock = open("/tmp/cumotion-roundtrip.lock", "w")
fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
rclpy.init()
node = rclpy.create_node("cumotion_direction_demo")
active = None
states = {}

def receive(key, msg):
    states[key] = (time.monotonic(), msg)

subs = [
    node.create_subscription(
        JointState, topic, lambda msg, key=key: receive(key, msg),
        qos_profile_sensor_data)
    for key, topic in [
        ("control", "/joint_states"), ("sim", "/isaac_joint_states")]
]

def wait(future, seconds):
    end = time.monotonic() + seconds
    while not future.done() and time.monotonic() < end:
        rclpy.spin_once(node, timeout_sec=0.1)
    if not future.done():
        raise TimeoutError("응답 시간 초과")
    return future.result()

def fresh(key, seconds=10):
    since = time.monotonic()
    while time.monotonic() - since < seconds:
        rclpy.spin_once(node, timeout_sec=0.1)
        record = states.get(key)
        if record and record[0] >= since:
            msg = record[1]
            if len(msg.name) == len(msg.position) and all(n in msg.name for n in ARM):
                return copy.deepcopy(msg)
    raise RuntimeError(f"{key}: 새 관절 데이터 수신 실패")

def action(client, goal):
    global active
    active = wait(client.send_goal_async(goal), 15)
    if not active.accepted:
        active = None
        raise RuntimeError("요청이 거절됐습니다.")
    result = wait(active.get_result_async(), 90)
    active = None
    if result.status != 4 or result.result.error_code.val != 1:
        raise RuntimeError(
            f"요청 실패: 상태={result.status}, 코드={result.result.error_code.val}")
    return result.result

try:
    service = node.create_client(GetParameters, "/cumotion_planner/get_parameters")
    if not service.wait_for_service(timeout_sec=10):
        raise RuntimeError("cuMotion 설정 서버가 없습니다.")
    query = GetParameters.Request()
    query.names = ["xrdf_file_path"]
    response = wait(service.call_async(query), 10)
    config = yaml.safe_load(Path(response.values[0].string_value).read_text())
    tools = config.get("tool_frames", [])
    if not tools or not isinstance(tools[0], str):
        raise RuntimeError("XRDF 손끝 프레임 형식 확인이 필요합니다.")
    TOOL, FRAME = tools[0], "panda_link0"
    print(f"기준: {FRAME} / 손끝: {TOOL}", flush=True)
    print("상하는 Z축, 좌우는 +Y/-Y축입니다. 카메라 방향과 다를 수 있습니다.", flush=True)
    fk_client = node.create_client(GetPositionFK, "/compute_fk")
    planner = ActionClient(node, MoveGroup, "/move_action")
    executor = ActionClient(node, ExecuteTrajectory, "/execute_trajectory")
    if not fk_client.wait_for_service(timeout_sec=10):
        raise RuntimeError("손끝 위치 계산 서비스가 없습니다.")
    if not planner.wait_for_server(timeout_sec=15) or not executor.wait_for_server(timeout_sec=15):
        raise RuntimeError("계획 또는 실행 서버가 준비되지 않았습니다.")

    def fk(msg):
        query = GetPositionFK.Request()
        query.header.frame_id = FRAME
        query.fk_link_names = [TOOL]
        query.robot_state.joint_state = msg
        query.robot_state.is_diff = False
        result = wait(fk_client.call_async(query), 10)
        if result.error_code.val != 1 or len(result.pose_stamped) != 1:
            raise RuntimeError(f"손끝 계산 실패: {result.error_code.val}")
        return result.pose_stamped[0].pose

    def errors(actual, target):
        a, b = actual.position, target.position
        distance = math.sqrt((a.x-b.x)**2+(a.y-b.y)**2+(a.z-b.z)**2)
        a, b = actual.orientation, target.orientation
        q, r = [a.x,a.y,a.z,a.w], [b.x,b.y,b.z,b.w]
        norm = math.sqrt(sum(v*v for v in q)*sum(v*v for v in r))
        if norm < 1e-10:
            raise RuntimeError("손끝 방향 값이 올바르지 않습니다.")
        angle = 2*math.acos(min(1.0, abs(sum(x*y for x,y in zip(q,r))/norm)))
        return distance, angle

    anchor = fk(fresh("sim"))
    d, angle = errors(fk(fresh("control")), anchor)
    if d > 0.015 or angle > 0.10:
        raise RuntimeError("제어기와 Isaac Sim 손끝 위치가 다릅니다.")
    print(f"기준 위치: X={anchor.position.x:.3f}, Y={anchor.position.y:.3f}, Z={anchor.position.z:.3f} m", flush=True)
    sequence = [
        ("위",0,DISTANCE), ("기준",0,0),
        ("아래",0,-DISTANCE), ("기준",0,0),
        ("좌(+Y)",DISTANCE,0), ("기준",0,0),
        ("우(-Y)",-DISTANCE,0), ("기준",0,0)]
    for cycle in range(1, REPEAT+1):
        for label, dy, dz in sequence:
            target = copy.deepcopy(anchor)
            target.position.y += dy
            target.position.z += dz
            goal = MoveGroup.Goal()
            req = goal.request
            req.group_name = "panda_arm"
            req.pipeline_id = "isaac_ros_cumotion"
            req.planner_id = "cuMotion"
            req.num_planning_attempts = 1
            req.allowed_planning_time = 10.0
            req.max_velocity_scaling_factor = 0.10
            req.max_acceleration_scaling_factor = 0.10
            req.start_state.joint_state = fresh("control")
            req.start_state.is_diff = False
            fresh("sim")
            constraint = Constraints()
            constraint.name = label
            pos = PositionConstraint()
            pos.header.frame_id = FRAME
            pos.link_name = TOOL
            pos.weight = 1.0
            sphere = SolidPrimitive()
            sphere.type = SolidPrimitive.SPHERE
            sphere.dimensions = [0.003]
            pos.constraint_region.primitives = [sphere]
            region = copy.deepcopy(target)
            region.orientation.x = region.orientation.y = region.orientation.z = 0.0
            region.orientation.w = 1.0
            pos.constraint_region.primitive_poses = [region]
            orient = OrientationConstraint()
            orient.header.frame_id = FRAME
            orient.link_name = TOOL
            orient.orientation = copy.deepcopy(target.orientation)
            orient.absolute_x_axis_tolerance = 0.03
            orient.absolute_y_axis_tolerance = 0.03
            orient.absolute_z_axis_tolerance = 0.03
            orient.weight = 1.0
            constraint.position_constraints = [pos]
            constraint.orientation_constraints = [orient]
            req.goal_constraints = [constraint]
            goal.planning_options.plan_only = True
            goal.planning_options.replan = False
            goal.planning_options.planning_scene_diff.is_diff = True
            goal.planning_options.planning_scene_diff.robot_state.is_diff = True
            print(f"[{cycle}/{REPEAT}] {label}: cuMotion 계획 중", flush=True)
            trajectory = action(planner, goal).planned_trajectory
            jt = trajectory.joint_trajectory
            if not jt.points or len(jt.joint_names) != len(jt.points[-1].positions):
                raise RuntimeError("유효한 경로가 없습니다.")
            if not all(n in jt.joint_names for n in ARM):
                raise RuntimeError("경로에 팔 관절이 누락됐습니다.")
            endpoint = copy.deepcopy(req.start_state.joint_state)
            values = dict(zip(endpoint.name, endpoint.position))
            values.update(zip(jt.joint_names, jt.points[-1].positions))
            endpoint.position = [float(values[n]) for n in endpoint.name]
            endpoint.velocity = []
            endpoint.effort = []
            d, angle = errors(fk(endpoint), target)
            if d > 0.008 or angle > 0.08:
                raise RuntimeError(f"계획 끝점 불일치: {d*1000:.1f} mm / {angle:.3f} rad")
            print(f"  {len(jt.points)}개 경로점 확인 / 실행 중", flush=True)
            execute = ExecuteTrajectory.Goal()
            execute.trajectory = trajectory
            action(executor, execute)
            deadline = time.monotonic()+20
            while True:
                d, angle = errors(fk(fresh("sim",3)), target)
                if d <= 0.012 and angle <= 0.10:
                    break
                if time.monotonic() >= deadline:
                    raise RuntimeError(f"실제 손끝 미도달: {d*1000:.1f} mm / {angle:.3f} rad")
            print(f"  {label} 도착: 위치 오차 {d*1000:.1f} mm / 방향 오차 {angle:.3f} rad", flush=True)
            time.sleep(2)
    print("상하좌우 시연 완료! 시작 손끝 위치로 복귀했습니다.", flush=True)
finally:
    if active is not None and active.accepted:
        try:
            wait(active.cancel_goal_async(), 5)
            print("진행 중 요청에 취소를 요청했습니다.", flush=True)
        except Exception as error:
            print("취소 확인 실패:", error, flush=True)
    node.destroy_node()
    rclpy.shutdown()
    lock.close()
'''
assert 0.01 <= DISTANCE_M <= 0.05, "이동 거리는 1~5cm로 설정하세요."
assert isinstance(REPEAT, int) and 1 <= REPEAT <= 3, "반복 횟수는 1~3으로 설정하세요."
robot_script = robot_script.replace("__REPEAT__", str(REPEAT)).replace("__DISTANCE__", repr(DISTANCE_M))

command = r'''
set -e
source /opt/ros/jazzy/setup.bash
source /workspaces/isaac_ros-dev/control-overlay/install/setup.bash
export RMW_IMPLEMENTATION=rmw_fastrtps_cpp
export FASTRTPS_DEFAULT_PROFILES_FILE=/workspaces/isaac_ros-dev/dds-udp.xml
export FASTDDS_DEFAULT_PROFILES_FILE="$FASTRTPS_DEFAULT_PROFILES_FILE"
exec timeout --signal=INT --kill-after=10s 600s python3 -u -
'''
p = subprocess.Popen(
    ["sudo", "-n", "docker", "exec", "-i", "cumotion-desktop", "bash", "-lc", command],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True)
p.stdin.write(robot_script)
p.stdin.close()
for line in p.stdout:
    print(line, end="", flush=True)
print("\n종료 코드:", p.wait())
